# 03 - IV Surface

Heatmap of IV across **strike × expiry**.

Rows = expiry dates, columns = strikes, color = IV%.


## Setup

In [ ]:
import sys, sqlite3
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_p = Path.cwd().resolve()
while _p != _p.parent and not (_p / "database").exists():
    _p = _p.parent
sys.path.insert(0, str(_p))

OPT_DB = _p / "database" / "options.db"
MKT_DB = _p / "database" / "market.db"

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 60)
plt.rcParams["figure.figsize"] = (13, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

def opt(q, **p):
    with sqlite3.connect(OPT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

def mkt(q, **p):
    with sqlite3.connect(MKT_DB) as c:
        return pd.read_sql_query(q, c, params=p)

print("options.db:", OPT_DB)
print("market.db :", MKT_DB)

## 1. Load IV snapshot

In [ ]:
iv = opt("""
    SELECT * FROM implied_volatility
    WHERE underlying='ETH'
      AND ts = (SELECT MAX(ts) FROM implied_volatility WHERE underlying='ETH')
""")
print("rows:", len(iv))
iv["dte"] = iv.dte.astype(float)
iv = iv[(iv.dte > 0) & (iv.dte < 200)]

## 2. IV surface heatmap

In [ ]:
if not iv.empty:
    pivot = iv.pivot_table(index="expiry", columns="strike",
                           values="iv", aggfunc="mean")
    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(13, 6))
        im = ax.imshow(pivot.values * 100, aspect="auto", cmap="viridis")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([pd.to_datetime(e, unit="ms").date() for e in pivot.index])
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([f"{int(s)}" for s in pivot.columns], rotation=45)
        plt.colorbar(im, ax=ax, label="IV %")
        ax.set_title("IV surface — strikes × expiries")
        ax.set_xlabel("strike")
        ax.set_ylabel("expiry")
        plt.tight_layout()
        plt.show()

## 3. Term structure (ATM IV per expiry)

In [ ]:
if not iv.empty:
    # ATM IV per expiry = strike nearest underlying
    oc = opt("SELECT underlying_px FROM option_chain ORDER BY ts DESC LIMIT 1")
    spot = float(oc.underlying_px.iloc[0]) if not oc.empty else None

    if spot:
        rows = []
        for e, g in iv.groupby("expiry"):
            atm = g.iloc[(g.strike - spot).abs().argsort()[:1]]
            dte = float(g.dte.iloc[0])
            rows.append({"expiry": pd.to_datetime(e, unit="ms").date(),
                         "dte": dte, "atm_iv": float(atm.iv.iloc[0])})
        ts_df = pd.DataFrame(rows).sort_values("dte")

        fig, ax = plt.subplots()
        ax.plot(ts_df.dte, ts_df.atm_iv * 100, "o-", color="purple")
        ax.set_title("ATM IV term structure")
        ax.set_xlabel("days to expiry")
        ax.set_ylabel("IV %")
        plt.tight_layout()
        plt.show()
        ts_df